In [1]:
import gymnasium as gym
import tensorflow as tf
import numpy as np
from collections import deque
import random


In [2]:
env = gym.make("CartPole-v1")
input_shape = [4]
n_outputs = 2

In [3]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=input_shape),
    tf.keras.layers.Dense(32, activation="elu"),
    tf.keras.layers.Dense(32, activation="elu"),
    tf.keras.layers.Dense(n_outputs)
])


In [4]:
target = tf.keras.models.clone_model(model)
target.set_weights(model.get_weights())

In [5]:
batch_size = 32
discount_factor = 0.95
learning_rate = 1e-3

optimizer = tf.keras.optimizers.Adam(
    learning_rate=learning_rate
)

loss_fn = tf.keras.losses.MeanSquaredError()



In [6]:
replay_buffer = deque(maxlen=2000)

In [7]:
def epsilon_greedy_policy(state, epsilon=0.0):
    if np.random.rand() < epsilon:
        return np.random.randint(n_outputs)

    q_values = model.predict(
        state[np.newaxis],
        verbose=0
    )

    return np.argmax(q_values[0])

#

In [8]:
def play_one_step(env, state, epsilon):

    action = epsilon_greedy_policy(
        state,
        epsilon
    )

    next_state, reward, terminated, truncated, info = env.step(action)

    done = terminated or truncated

    replay_buffer.append(
        (state, action, reward, next_state, done)
    )

    return next_state, reward, done, info



In [9]:
def sample_experiences(batch_size):

    indices = np.random.randint(
        len(replay_buffer),
        size=batch_size
    )

    batch = [
        replay_buffer[index]
        for index in indices
    ]

    states, actions, rewards, next_states, dones = [
        np.array(
            [experience[field_index]
             for experience in batch]
        )
        for field_index in range(5)
    ]

    return (
        states,
        actions,
        rewards,
        next_states,
        dones
    )

In [10]:
def training_step(batch_size):

    states, actions, rewards, next_states, dones = \
        sample_experiences(batch_size)


    next_q_values = target.predict(
        next_states,
        verbose=0
    )

    max_next_q_values = np.max(
        next_q_values,
        axis=1
    )

    target_q_values = rewards + \
        (1 - dones.astype(np.float32)) * \
        discount_factor * \
        max_next_q_values

    target_q_values = target_q_values.reshape(-1, 1)

    mask = tf.one_hot(
        actions,
        n_outputs
    )

    with tf.GradientTape() as tape:

        all_q_values = model(states)

        q_values = tf.reduce_sum(
            all_q_values * mask,
            axis=1,
            keepdims=True
        )

        loss = loss_fn(
            target_q_values,
            q_values
        )

    grads = tape.gradient(
        loss,
        model.trainable_variables
    )

    optimizer.apply_gradients(
        zip(
            grads,
            model.trainable_variables
        )
    )

In [ ]:
episodes = 600
target_update_freq = 50

for episode in range(episodes):

    state, info = env.reset()

    total_reward = 0

    epsilon = max(
        1 - episode / 500,
        0.01
    )

    for step in range(200):

        state, reward, done, info = play_one_step(
            env,
            state,
            epsilon
        )

        total_reward += reward

        if len(replay_buffer) >= batch_size:
            training_step(batch_size)

        if done:
            break

    if episode % target_update_freq == 0:
        target.set_weights(
            model.get_weights()
        )

    if episode % 10 == 0:
        print(
            f"Episode {episode:3d} | "
            f"Reward: {total_reward:3.0f} | "
            f"Epsilon: {epsilon:.3f}"
        )


state, info = env.reset()

done = False
total_reward = 0

while not done:

    q_values = model.predict(
        state[np.newaxis],
        verbose=0
    )

    action = np.argmax(q_values[0])

    state, reward, terminated, truncated, info = env.step(action)

    done = terminated or truncated

    total_reward += reward

print("\nTest Reward:", total_reward)

env.close() 

Episode   0 | Reward:  26 | Epsilon: 1.000
Episode  10 | Reward:  19 | Epsilon: 0.980
Episode  20 | Reward:  11 | Epsilon: 0.960
Episode  30 | Reward:  20 | Epsilon: 0.940
Episode  40 | Reward:  36 | Epsilon: 0.920
Episode  50 | Reward:  10 | Epsilon: 0.900
Episode  60 | Reward:  11 | Epsilon: 0.880
Episode  70 | Reward:   9 | Epsilon: 0.860
Episode  80 | Reward:  34 | Epsilon: 0.840
Episode  90 | Reward:  25 | Epsilon: 0.820
Episode 100 | Reward:  16 | Epsilon: 0.800
Episode 110 | Reward:  12 | Epsilon: 0.780
Episode 120 | Reward:  16 | Epsilon: 0.760
Episode 130 | Reward:  15 | Epsilon: 0.740
Episode 140 | Reward:  32 | Epsilon: 0.720
Episode 150 | Reward:  53 | Epsilon: 0.700
Episode 160 | Reward:  15 | Epsilon: 0.680
Episode 170 | Reward:  16 | Epsilon: 0.660
Episode 180 | Reward:  14 | Epsilon: 0.640
Episode 190 | Reward:  18 | Epsilon: 0.620
